# Attention Value Routing

Analyze how attention routes value information: source routing,
output decomposition, diversity, logit effects, and summary.

## Why This Matters

Attention value routing reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_value_routing import (
    value_source_routing, value_output_decomposition,
    value_routing_diversity, value_logit_routing,
    value_routing_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Source Routing

Where does value information come from for a specific head and query?

In [ ]:
result = value_source_routing(model, tokens, layer=0, head=0)
for s in result['per_source'][:5]:
    bar = '█' * int(s['weighted_contribution'] * 50)
    print(f"  Pos {s['position']} (tok {s['token_id']:3d}): "
          f"attn={s['attention_weight']:.4f}, v_norm={s['value_norm']:.4f}, "
          f"contrib={s['weighted_contribution']:.4f} {bar}")

## Output Decomposition

Decompose head output into per-source contributions through OV circuit.

In [ ]:
result = value_output_decomposition(model, tokens, layer=0, head=0)
print(f"Total output norm: {result['total_output_norm']:.4f}\n")
for s in result['per_source'][:5]:
    print(f"  Pos {s['position']}: norm={s['output_norm']:.4f}, "
          f"frac={s['fraction_of_total']:.2%}, align={s['alignment_with_total']:.4f}")

## Routing Diversity

Are heads routing diverse information or copying each other?

In [ ]:
for layer in range(4):
    result = value_routing_diversity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    print(f"  Layer {layer}: mean_sim={result['mean_output_similarity']:.4f}{flag}")

## Logit Routing

What tokens does the routed value promote in the vocabulary?

In [ ]:
for head in range(4):
    result = value_logit_routing(model, tokens, layer=0, head=head)
    print(f"  L0H{head}: promotes tok {result['top_promoted']:3d} "
          f"({result['top_promoted_logit']:+.4f}), "
          f"suppresses tok {result['top_suppressed']:3d} "
          f"({result['top_suppressed_logit']:+.4f}), range={result['logit_range']:.4f}")

## Routing Summary

Cross-layer overview of value routing output magnitudes.

In [ ]:
result = value_routing_summary(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(min(p['mean_output_norm'] * 20, 20))
    print(f"  Layer {p['layer']}: mean={p['mean_output_norm']:.4f}, "
          f"max={p['max_output_norm']:.4f}, dominant=H{p['dominant_head']} {bar}")